基于规则的关系抽取

In [1]:
import re

# 示例：简单的规则匹配
text = "马云创立了阿里巴巴"
pattern = r"(.+?)创立了(.+?)"
match = re.search(pattern, text)

if match:
    print(f"创始人: {match.group(1)}, 公司: {match.group(2)}")

创始人: 马云, 公司: 阿


使用 spaCy 进行关系抽取

In [2]:
import spacy

# 加载英文模型
nlp = spacy.load("en_core_web_sm")
'''import spacy 和 nlp = spacy.load("en_core_web_sm")：加载 spaCy 预训练好的轻量级英文模型。这个模型不仅懂英语语法，还具备实体识别能力。'''
text = "Apple was founded by Steve Jobs in 1976."
doc = nlp(text)

# 识别命名实体
for ent in doc.ents:#for ent in doc.ents:：遍历模型识别出来的 所有命名实体（Entities），ent 代表每一个实体。
    print(ent.text, ent.label_)
#PERSON 代表人名。

Apple ORG
Steve Jobs PERSON
1976 DATE


使用 Hugging Face Transformers 进行文本分类

In [2]:
import os
'''使用 Hugging Face transformers 库，基于本地微调过的 BERT 模型，进行中文电商评论情感二分类 '''
from transformers import (AutoTokenizer,AutoModelForSequenceClassification,pipeline)

# 1. 设置本地中文文本分类模型路径
model_path = r"D:\11\NLP\data\jd-sentiment-local"

# 2. 检查本地模型文件夹
if not os.path.isdir(model_path):
    raise FileNotFoundError(f"找不到模型文件夹：\n{model_path}")


# 3. 从本地加载分词器
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)


# 4. 从本地加载文本分类模型
model = AutoModelForSequenceClassification.from_pretrained(model_path,local_files_only=True)

# 5. 创建文本分类pipeline
classifier = pipeline(task="text-classification",model=model,tokenizer=tokenizer,device=-1)
'''device=-1：pipeline 中的这个参数非常关键，它告诉模型强制使用 CPU 进行推理。即使你的电脑有显卡（GPU），也会强制跑在 CPU 上。这在做小规模测试时非常有用（不占用显卡显存）。'''

# 6. 准备测试文本
texts = [
    "这个手机运行速度很快，拍照效果也很好，我很满意。",
    "商品质量很差，刚使用一天就坏了。",
    "物流很快，包装也很完整。",
    "客服态度不好，问题一直没有得到解决。"
]

# 7. 执行文本分类
results = classifier(texts,truncation=True,max_length=512)

# 8. 显示分类结果
for text, result in zip(texts, results):
    original_label = result["label"]
    score = result["score"]
    # 把英文标签转换为中文
    if "negative" in original_label.lower():
        chinese_label = "负面"
    elif "positive" in original_label.lower():
        chinese_label = "正面"
    else:
        chinese_label = original_label

    print(f"文本：{text}")
    print(f"分类结果：{chinese_label}")
    print(f"原始标签：{original_label}")
    print(f"置信度：{score:.4f}")
    print("-" * 60)

Device set to use cpu


文本：这个手机运行速度很快，拍照效果也很好，我很满意。
分类结果：正面
原始标签：positive (stars 4 and 5)
置信度：0.9937
------------------------------------------------------------
文本：商品质量很差，刚使用一天就坏了。
分类结果：负面
原始标签：negative (stars 1, 2 and 3)
置信度：0.9847
------------------------------------------------------------
文本：物流很快，包装也很完整。
分类结果：正面
原始标签：positive (stars 4 and 5)
置信度：0.9918
------------------------------------------------------------
文本：客服态度不好，问题一直没有得到解决。
分类结果：负面
原始标签：negative (stars 1, 2 and 3)
置信度：0.9723
------------------------------------------------------------


数据准备（示例数据集）

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
#用传统机器学习（TF-IDF + SVM）来解决中文的短文本关系分类问题。

# 1. 准备训练数据
train_data = [
    # 创始人关系
    ("马云创立了阿里巴巴", "创始人"),
    ("比尔盖茨是微软的创始人", "创始人"),
    ("李彦宏创立了百度公司", "创始人"),
    ("马化腾创建了腾讯公司", "创始人"),
    ("任正非创办了华为公司", "创始人"),
    ("雷军是小米公司的创始人", "创始人"),

    # 首都关系
    ("北京是中国的首都", "首都"),
    ("东京是日本的首都", "首都"),
    ("巴黎是法国的首都", "首都"),
    ("伦敦是英国的首都", "首都"),
    ("首尔是韩国的首都", "首都"),
    ("柏林是德国的首都", "首都"),
]


# ==================================================
# 2. 分离文本和标签
# ==================================================

texts = [item[0] for item in train_data]
labels = [item[1] for item in train_data]


# ==================================================
# 3. 字符级TF-IDF
# ==================================================

vectorizer = TfidfVectorizer(
    analyzer="char",        # 按字符切分，适合简单中文演示
    ngram_range=(2, 4)      # 提取2到4个连续字符
)
'''ngram_range=(2, 4)：意思是提取2个字、3个字、4个字的连续片段。
实际举例：假设训练句是 "马云创立了阿里巴巴"。
TF-IDF 会提取出：马云、云创、创立、立了、了阿、阿里、里巴、巴巴、马云创立、云创立了、创立了阿、了阿里巴……等等几百上千个特征。
当模型训练完成后，如果新的测试句里有重复的字符片段，它就会根据这些特征给予打分。'''
X = vectorizer.fit_transform(texts)

print("训练数据数量：", X.shape[0])
print("特征数量：", X.shape[1])


# ==================================================
# 4. 训练线性SVM
# ==================================================

svm_model = LinearSVC()
'''LinearSVC()：这是一个线性的 SVM 分类器。
它的数学原理：TF-IDF 把句子变成了几百维的数学向量，LinearSVC 的动作就像是在这个多维空间里画一条完美的分界线。把代表“创始人”的句子和代表“首都”的句子在数学空间里分开。'''
svm_model.fit(X, labels)
print("模型训练完成")
print("支持的关系类别：", svm_model.classes_)


# ==================================================
# 5. 测试新文本
# ==================================================
test_texts = [
    "乔布斯创立了苹果公司",
    "罗马是意大利的首都",
    "扎克伯格是Facebook的创始人"
]

test_vectors = vectorizer.transform(test_texts)
predictions = svm_model.predict(test_vectors)

# ==================================================
# 6. 输出结果
# ==================================================
for text, prediction in zip(test_texts, predictions):
    print(f"测试文本：{text}")
    print(f"预测关系：{prediction}")
    print("-" * 50)

训练数据数量： 12
特征数量： 215
模型训练完成
支持的关系类别： ['创始人' '首都']
测试文本：乔布斯创立了苹果公司
预测关系：创始人
--------------------------------------------------
测试文本：罗马是意大利的首都
预测关系：首都
--------------------------------------------------
测试文本：扎克伯格是Facebook的创始人
预测关系：创始人
--------------------------------------------------


补充 1：更完整的基于规则的关系抽取

In [5]:
import re


def extract_relations(text):
    relations = []

    # 清除首尾空格以及句末标点
    text = text.strip().rstrip("。！？")
    #把句子首尾的空格、句号、感叹号、问号都删掉了。
    patterns = [
        # 创始人关系
        (r"(.+?)创立了(.+)", "创始人"),
        (r"(.+?)是(.+?)的创始人", "创始人"),

        # 首都关系
        (r"(.+?)是(.+?)的首都", "首都"),

        # 位于关系
        (r"(.+?)位于(.+)", "位于"),

        # 任职关系
        (r"(.+?)担任(.+?)的(.+)", "任职"),
    ]

    for pattern, relation_type in patterns:

        # fullmatch 要求整个文本都符合正则表达式
        match = re.fullmatch(pattern, text)

        if match is None:
            continue

        groups = match.groups()

        if len(groups) == 2:
            relations.append({
                "head": groups[0].strip(),
                "tail": groups[1].strip(),
                "type": relation_type
            })

        elif len(groups) == 3:
            relations.append({
                "head": groups[0].strip(),
                "tail": groups[1].strip(),
                "type": f"{relation_type}: {groups[2].strip()}"
            })

    return relations


# 测试文本
test_texts = [
    "马云创立了阿里巴巴",
    "北京是中国的首都",
    "张一鸣担任字节跳动的CEO"
]

for text in test_texts:
    result = extract_relations(text)

    print(f"文本：{text}")
    print(f"抽取关系：{result}")
    print()

文本：马云创立了阿里巴巴
抽取关系：[{'head': '马云', 'tail': '阿里巴巴', 'type': '创始人'}]

文本：北京是中国的首都
抽取关系：[{'head': '北京', 'tail': '中国', 'type': '首都'}]

文本：张一鸣担任字节跳动的CEO
抽取关系：[{'head': '张一鸣', 'tail': '字节跳动', 'type': '任职: CEO'}]



补充 2：使用 spaCy 进行关系抽取（规则+依存句法）

In [4]:
import spacy
#spaCy 的依存句法分析，从英文被动句中提取“创始人—公司”关系。
nlp = spacy.load("en_core_web_sm")

def extract_relations_spacy_fixed(text):
    doc = nlp(text)
    relations = []

    for token in doc:
        if token.lemma_ == "found" and token.dep_ == "ROOT":#找到一个原形为 found，并且是句子核心谓语的动词。
            # 提取被动主语（公司）
            subjects = [child for child in token.children if child.dep_ == "nsubjpass"]

            # 遍历子节点，提取施事者（创始人）
            for child in token.children:
                if child.dep_ == "agent":
                    # child是 "by" 这个词，需要找它后面的介词宾语(pobj)
                    for grandchild in child.children:
                        if grandchild.dep_ == "pobj":
                            # 成功提取到真正的创始人
                            for subj in subjects:
                                relations.append({
                                    "head": grandchild.text,  # Steve Jobs
                                    "tail": subj.text,        # Apple
                                    "type": "founder"
                                })
    return relations

# 测试
text = "Apple was founded by Steve Jobs in 1976."
results = extract_relations_spacy_fixed(text)
print(results)

[{'head': 'Jobs', 'tail': 'Apple', 'type': 'founder'}]


补充 3：使用预训练关系抽取模型（REBEL）

In [1]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM
)


# ==================================================
# 1. 设置本地REBEL模型路径
# ==================================================

model_path = r"D:\11\NLP\data\rebel-large-local"


# ==================================================
# 2. 检查模型目录
# ==================================================

if not os.path.isdir(model_path):
    raise FileNotFoundError(
        f"找不到REBEL本地模型目录：\n{model_path}"
    )

required_files = [
    "config.json",
    "model.safetensors",
    "added_tokens.json",
    "merges.txt",
    "special_tokens_map.json",
    "tokenizer.json",
    "tokenizer_config.json",
    "vocab.json"
]

missing_files = []

for file_name in required_files:
    file_path = os.path.join(model_path, file_name)

    if not os.path.isfile(file_path):
        missing_files.append(file_name)

if missing_files:
    raise FileNotFoundError(
        "本地REBEL模型缺少以下文件：\n"
        + "\n".join(missing_files)
    )

print("本地REBEL模型文件检查成功。")
#REBEL 是一个用于关系抽取的预训练模型。

# ==================================================
# 3. 只从本地加载分词器和模型
# ==================================================
tokenizer = AutoTokenizer.from_pretrained(model_path,local_files_only=True)
model = AutoModelForSeq2SeqLM.from_pretrained(model_path,local_files_only=True)

# ==================================================
# 4. 设置运行设备
# ==================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
model.eval()
print("模型运行设备：", device)

# ==================================================
# 5. 解析REBEL生成结果
# ==================================================
def parse_rebel_output(generated_text: str) -> list[dict]:
    """
    将REBEL生成的特殊格式文本解析为关系三元组。
    返回格式：
    [
        {
            "head": "Steve Jobs",
            "relation": "founder",
            "tail": "Apple Inc."
        }
    ]
    """
    triplets = []
    subject = ""
    relation = ""
    object_ = ""
    current_part = None

    # 删除普通的句子开始、结束和填充标记
    cleaned_text = (generated_text.replace("<s>", "").replace("</s>", "").replace("<pad>", "").strip())

    for token in cleaned_text.split():
        # 一个新三元组开始
        if token == "<triplet>":
            # 保存前一个完整三元组
            if subject and relation and object_:
                triplets.append({"head": subject.strip(),"relation": relation.strip(),"tail": object_.strip()})
            subject = ""
            relation = ""
            object_ = ""
            current_part = "subject"
        # REBEL中的<subj>后面实际上是宾语内容
        elif token == "<subj>":
            current_part = "object"
        # REBEL中的<obj>后面实际上是关系类型
        elif token == "<obj>":
            current_part = "relation"
        else:
            if current_part == "subject":
                subject += " " + token
            elif current_part == "object":
                object_ += " " + token
            elif current_part == "relation":
                relation += " " + token
    # 保存最后一个三元组
    if subject and relation and object_:
        triplets.append({"head": subject.strip(),"relation": relation.strip(), "tail": object_.strip() })
    return triplets

# ==================================================
# 6. 定义关系抽取函数
# ==================================================

def extract_relations_rebel(text: str) -> tuple[str, list[dict]]:
    """
    使用本地REBEL模型从英文文本中抽取关系。
    """
    if not isinstance(text, str) or not text.strip():
        raise ValueError("输入文本不能为空。")
    # 将文本转换为模型输入
    model_inputs = tokenizer( text,max_length=256,truncation=True, return_tensors="pt")

    # 将输入放到模型所在设备
    input_ids = model_inputs["input_ids"].to(device)
    attention_mask = model_inputs["attention_mask"].to(device)

    # 关闭梯度，提高推理效率
    with torch.no_grad():
        generated_tokens = model.generate(input_ids=input_ids,attention_mask=attention_mask, max_length=256, num_beams=3, num_return_sequences=1,length_penalty=0)
    # 必须保留<triplet>、<subj>、<obj>
    generated_text = tokenizer.decode(generated_tokens[0],skip_special_tokens=False)
    # 解析三元组
    triplets = parse_rebel_output(generated_text)
    return generated_text, triplets


# ==================================================
# 7. 测试
# ==================================================
text = "Steve Jobs was the co-founder of Apple Inc."
generated_text, relations = extract_relations_rebel(text)


# ==================================================
# 8. 输出结果
# ==================================================
print("\n原始文本：")
print(text)
print("\n模型生成的原始结果：")
print(generated_text)
print("\n抽取到的关系三元组：")
if not relations:
    print("没有识别到关系。")
else:
    for index, relation in enumerate(relations, start=1):
        print(f"\n关系 {index}")
        print(f"主体：{relation['head']}")
        print(f"关系：{relation['relation']}")
        print(f"客体：{relation['tail']}")
'''这段代码的作用是：把一段英文文本输入本地 REBEL 关系抽取模型，让模型自动识别文本中的实体和它们之间的关系。例如输入“Steve Jobs was the co-founder of Apple Inc.”，模型会先生成带有 `<triplet>`、`<subj>`、`<obj>` 等特殊标记的结果，然后代码再把这个结果解析成结构化的关系三元组，最终得到“Steve Jobs—founder—Apple Inc.”，也就是识别出“史蒂夫·乔布斯是苹果公司的创始人”这一关系。'''

C:\Users\Administrator\.conda\envs\rl\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


本地REBEL模型文件检查成功。
模型运行设备： cpu

原始文本：
Steve Jobs was the co-founder of Apple Inc.

模型生成的原始结果：
<s><triplet> Steve Jobs <subj> Apple Inc. <obj> employer <triplet> Apple Inc. <subj> Steve Jobs <obj> founded by</s>

抽取到的关系三元组：

关系 1
主体：Steve Jobs
关系：employer
客体：Apple Inc.

关系 2
主体：Apple Inc.
关系：founded by
客体：Steve Jobs
